# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and processing the FAIR² dataset on second primary colorectal cancer in cancer survivors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL, ensuring rich, standardized metadata and easy data access.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We will load the dataset metadata and records using the `mlcroissant` interface. This ensures that all subsequent steps are done using the standardized Croissant API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Let's review the available record sets in the dataset, their IDs, and fields. All entities will be referenced by their `@id` as required for Croissant compatibility.

In [ ]:
# List all available record sets with their @id and field @ids
from pprint import pprint

print("Available record sets in the dataset:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}")
    if 'fields' in rs and isinstance(rs['fields'], list):
        field_ids = [f["@id"] for f in rs["fields"] if isinstance(f, dict) and "@id" in f]
        print(f"    fields: {field_ids}")
    else:
        print(f"    fields: []")

Below, we'll examine a single record from each record set for a quick glance at the data structure. This is helpful in identifying the most relevant record sets and fields for further processing.

In [ ]:
# Print a single record from each set as an overview
for rs in metadata.record_sets:
    rs_id = rs['@id']
    print(f"\nRecord set: {rs_id}")
    try:
        rec_iter = dataset.records(record_set=rs_id)
        first_example = next(rec_iter)
        pprint(first_example)
        print(f"Fields (@ids): {list(first_example.keys())}")
    except StopIteration:
        print("  (no records)")
    except Exception as e:
        print(f"  Error reading records: {e}")

## 3. Data Extraction

Now we'll extract the records from all tabular record sets into pandas DataFrames for easy exploration and analysis. You'll see how to select and reference each DataFrame by its record set `@id`.

In [ ]:
# Gather tabular data from all record sets
dataframes = {}
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
        else:
            print(f"Record set {rs_id} contains no records.")
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

if dataframes:
    main_rs_id = list(dataframes.keys())[0]  # Picking the first loaded DataFrame for demonstration
    print(f"\nSample columns in {main_rs_id}:\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Next, let's perform some simple data processing, such as filtering, normalization, and grouping. We'll operate on the largest or primary record set. **All columns will be referenced by their `@id` as per the Croissant standard.**

In [ ]:
# Pick the main record set (modify as needed for your use case)
record_set_id = main_rs_id  # e.g., 'cr:RecordSet:second_primary_crc'
df = dataframes[record_set_id]

# Identify candidate numeric columns by inspecting dtypes and sample values
numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_columns:
    # Try to infer numeric columns by attempting conversion
    numeric_columns = []
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna().iloc[:10])
            numeric_columns.append(col)
        except Exception:
            continue

if numeric_columns:
    numeric_field_id = numeric_columns[0]  # Take the first numeric field
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field detected. Please adjust this code block.")

# For demo, set a threshold as the mean if the field exists
if numeric_columns:
    threshold = df[numeric_field_id].dropna().astype(float).mean()
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Use the first non-numeric field as groupby key if available
    group_fields = [col for col in df.columns if col != numeric_field_id]
    group_field_id = None
    for col in group_fields:
        if df[col].dtype == object:
            group_field_id = col
            break
    
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization

Let's visualize the distribution of the selected numeric variable and explore group differences (if grouping field found).

_For reproducibility, all data elements are referenced by their Croissant `@id` fields._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna().astype(float), kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id} (by @id)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook showcased how to load, overview, filter, normalize, group, and visualize the FAIR² colorectal cancer dataset using the Croissant schema via `mlcroissant`. 

- All entities (record sets, fields, columns) were referenced strictly by `@id` fields as enforced by the FAIR data and Croissant standards.
- The Croissant model streamlines multi-table dataset exploration and guarantees metadata provenance.

For deeper domain analyses (e.g., clinicopathological predictors or molecular markers), you can build further upon these base steps, referencing each data element by its immutable `@id`. Happy data science!